<a href="https://colab.research.google.com/github/kuds/mesozoic-labs/blob/main/notebooks/jax_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JAX/MJX Dinosaur Training (Colab)

Train a dinosaur species using **MuJoCo MJX** (JAX-accelerated physics) with a
from-scratch PPO implementation in pure JAX. MJX vectorises thousands of parallel
simulations on a single GPU, giving 10-100x speedups over CPU-based Gymnasium training.

**Requirements:** Colab GPU runtime (A100 recommended).

**Supported Species:**
- `trex` — T-Rex: balance → locomotion → bite
- `velociraptor` — Raptor: balance → locomotion → strike
- `brachiosaurus` — Brachio: balance → locomotion → food reach

Set `SPECIES` in the configuration cell below to choose.

In [ ]:
# Install dependencies and verify GPU
!pip install mujoco mujoco-mjx "jax[cuda12]" flax optax

import os
import subprocess

if subprocess.run("nvidia-smi").returncode:
    raise RuntimeError("GPU not found. Use a GPU Colab runtime.")

NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
        f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

os.environ["MUJOCO_GL"] = "egl"

import jax
import mujoco
from mujoco import mjx

print(f"JAX devices: {jax.devices()}")
print(f"MuJoCo: {mujoco.__version__}")
print("Setup complete.")

In [ ]:
# GPU diagnostics — run this anytime to check utilization
# (especially useful to run from a second terminal during training)
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu,power.draw'],
    capture_output=True, text=True,
    env={**__import__('os').environ, 'COLUMNS': '200'},
)
if result.returncode == 0:
    lines = result.stdout.strip().split('\n')
    for line in lines:
        parts = [p.strip() for p in line.split(',')]
        if len(parts) >= 7:
            print(f'GPU:         {parts[0]}')
            print(f'Utilization: {parts[1]} (compute)  {parts[2]} (memory)')
            print(f'Memory:      {parts[3]} / {parts[4]}')
            print(f'Temperature: {parts[5]}   Power: {parts[6]}')
        else:
            print(line)
else:
    print('nvidia-smi failed:', result.stderr)

# Tip: You can also run this from a Colab terminal:
#   watch -n 2 nvidia-smi


In [ ]:
# Clone mesozoic-labs and install with JAX extras
!git clone https://github.com/kuds/mesozoic-labs.git /content/mesozoic-labs 2>/dev/null || echo 'Already cloned'
!pip install -e "/content/mesozoic-labs[jax]" -q

# Ensure the repo root is on sys.path so `from environments.…` works
# even if the editable install did not fully resolve.
import sys
if '/content/mesozoic-labs' not in sys.path:
    sys.path.insert(0, '/content/mesozoic-labs')

from IPython.display import clear_output

clear_output()
print("mesozoic-labs[jax] installed.")

In [ ]:
import logging
import time

# Silence JAX compilation spam (tracing/compiling/XLA warnings every update)
logging.getLogger("jax._src.dispatch").setLevel(logging.ERROR)
logging.getLogger("jax._src.interpreters.pxla").setLevel(logging.ERROR)

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import mujoco
import numpy as np
import optax
from mujoco import mjx

# Shared modules from mesozoic-labs
from environments.shared.reward_functions import (
    reward_forward_velocity,
    reward_alive,
    reward_energy,
    reward_posture,
)
from environments.shared.jax_ppo import (
    PPOConfig,
    make_actor_critic,
    make_optimizer,
    sample_action,
    compute_gae,
    ppo_loss,
)
from environments.shared.jax_normalization import RunningMeanStd, normalize_obs, update_running_stats, decay_running_stats
from environments.shared.jax_eval import EvalConfig, evaluate_policy_cpu, check_stage_gate
from environments.shared.jax_viz import plot_training_curves, plot_locomotion_diagnostics, record_training_video
from environments.shared.mjx_utils import scale_action_jax
from environments.shared.obs_functions import SensorLayout, build_bipedal_obs

print(f"JAX {jax.__version__}, devices: {jax.devices()}")
print(f"MuJoCo {mujoco.__version__}")

# Verify GPU
assert jax.devices()[0].platform == "gpu", "No GPU detected — JAX is using CPU. Check runtime type."
print("GPU OK")

In [ ]:
# ============================================================
# USER CONFIGURATION
# ============================================================
SPECIES = "trex"  # Choose: "trex", "velociraptor", "brachiosaurus"
CURRENT_STAGE = 1  # Curriculum stage: 1=balance, 2=locomotion, 3=species-specific
USE_GOOGLE_DRIVE = True  # Set to True to save outputs to Google Drive (persistent across sessions)
VERBOSE = 1  # 0=eval/summary only, 1=periodic updates (default), 2=every update

# Resume from a previous checkpoint (set to a .pkl path, or None to start fresh)
RESUME_FROM = None  # e.g. "trex_jax_checkpoint_100.pkl"

# ============================================================
# Species Configuration (auto-resolved from SPECIES)
# ============================================================
_SPECIES_CFG = {
    "trex": {
        "xml_path": "/content/mesozoic-labs/environments/trex/assets/trex.xml",
        "root_body": "pelvis",
        "healthy_z_range": (0.4, 1.6),
        "target_z_default": 0.5,
        "stage_names": {1: "Balance", 2: "Locomotion", 3: "Bite"},
    },
    "velociraptor": {
        "xml_path": "/content/mesozoic-labs/environments/velociraptor/assets/raptor.xml",
        "root_body": "pelvis",
        "healthy_z_range": (0.3, 1.0),
        "target_z_default": 0.3,
        "stage_names": {1: "Balance", 2: "Locomotion", 3: "Strike"},
    },
    "brachiosaurus": {
        "xml_path": "/content/mesozoic-labs/environments/brachiosaurus/assets/brachiosaurus.xml",
        "root_body": "torso",
        "healthy_z_range": (1.0, 3.5),
        "target_z_default": 3.0,
        "stage_names": {1: "Balance", 2: "Locomotion", 3: "Food Reach"},
    },
}

assert SPECIES in _SPECIES_CFG, f"Unknown species: {SPECIES}. Choose from: {list(_SPECIES_CFG)}"
_cfg = _SPECIES_CFG[SPECIES]

# ============================================================
# Storage Configuration (stage-organized directory structure)
# ============================================================
from datetime import datetime
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    _STORAGE_ROOT = Path("/content/drive/MyDrive/mesozoic-labs/logs/jax_training")
else:
    _STORAGE_ROOT = Path("logs")

_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = _STORAGE_ROOT / SPECIES / f"jax_{_timestamp}"
STAGE_DIR = RUN_DIR / f"stage{CURRENT_STAGE}"
MODEL_DIR = STAGE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# OUTPUT_DIR kept for backward compat (points to stage dir)
OUTPUT_DIR = STAGE_DIR

print(f"Species:    {SPECIES}")
print(f"Stage:      {CURRENT_STAGE} ({_cfg['stage_names'].get(CURRENT_STAGE, '?')})")
print(f"Run dir:    {RUN_DIR}")
print(f"Stage dir:  {STAGE_DIR}")
print(f"Drive:      {USE_GOOGLE_DRIVE}")
print(f"Verbose:    {VERBOSE}")
if RESUME_FROM:
    print(f"Resuming from checkpoint: {RESUME_FROM}")

## 1. Load Model into MJX

In [ ]:
# Load the MJCF model for the selected species
xml_path = _cfg["xml_path"]
mj_model = mujoco.MjModel.from_xml_path(xml_path)
mj_data = mujoco.MjData(mj_model)

print(f"{SPECIES.title()} model loaded:")
print(f"  Bodies: {mj_model.nbody}")
print(f"  Joints: {mj_model.njnt}")
print(f"  Actuators (nu): {mj_model.nu}")
print(f"  qpos dim: {mj_model.nq}")
print(f"  qvel dim: {mj_model.nv}")
print(f"  Geoms: {mj_model.ngeom}")
print(f"  Total mass: {sum(mj_model.body_mass):.2f} kg")
print(f"  Timestep: {mj_model.opt.timestep * 1000:.1f} ms")

# ==================== MJX Solver Tuning for GPU Throughput ====================
# MuJoCo defaults are tuned for CPU accuracy; MJX on GPU benefits from
# explicit solver configuration.  These settings reduce contact solver
# iterations and use CG for speed while keeping bipedal balance stable.

# Solver: CG (conjugate gradient) is faster than default Newton on GPU
# because it avoids the matrix factorization that Newton requires.
mj_model.opt.solver = mujoco.mjtSolver.mjSOL_CG

# Solver iterations: default is 100 (way too many for RL).
# For bipedal balance/locomotion, 4-6 CG iterations are sufficient.
# The warmstart flag in the XML helps CG converge in fewer iterations.
mj_model.opt.iterations = 4

# LS iterations (line search within solver): default 50, reduce to 4.
mj_model.opt.ls_iterations = 4

# Disable unnecessary computation flags for training speed:
# - filterparent: MJX doesn't benefit from this (parent-child contacts
#   are already handled by <exclude> pairs in the XML)
mj_model.opt.disableflags |= mujoco.mjtDisableBit.mjDSBL_FILTERPARENT

print(f"\nMJX solver tuning:")
print(f"  Solver: CG (conjugate gradient)")
print(f"  Iterations: {mj_model.opt.iterations}")
print(f"  LS iterations: {mj_model.opt.ls_iterations}")
print(f"  Disabled flags: filterparent")

# Put model on device (GPU)
mjx_model = mjx.put_model(mj_model)
print(f"\nModel placed on {jax.devices()[0]}")


## 2. Environment Functions
Pure-JAX functions for observation, reward, reset, and step. These now use
the **shared modules** from `environments.shared` so that the same reward and
observation logic is used by both the Gymnasium (SB3) and MJX (JAX) training
paths.

**Note:** The shared `reward_functions` use `float()` / Python `if` which break
`jax.vmap` tracing. The reward and termination functions below are re-implemented
in pure JAX (`jnp` only) for vmap compatibility.

In [ ]:
# ---------- Constants ----------
FRAME_SKIP = 5
HEALTHY_Z_MIN, HEALTHY_Z_MAX = _cfg["healthy_z_range"]
MAX_TILT_ANGLE = 1.047
MAX_EPISODE_STEPS = 1000

# Body / geom / site IDs (looked up once from the MuJoCo model)
ROOT_BODY_ID = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, _cfg["root_body"])
FLOOR_GEOM_ID = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_GEOM, "floor")

# Sensor layout (matches MJCF): gyro(3), accel(3), quat(4), foot sensors...
# Bipedal species have 2 foot sensors at indices 10-11
# Quadrupedal (brachiosaurus) has 4 foot sensors at indices 10-13
if SPECIES == "brachiosaurus":
    SENSOR_LAYOUT = SensorLayout(gyro_start=0, accel_start=3, quat_start=6, foot_indices=(10, 11, 12, 13))
else:
    SENSOR_LAYOUT = SensorLayout(gyro_start=0, accel_start=3, quat_start=6, foot_indices=(10, 11))

# Action scaling ranges (as JAX arrays for use with shared scale_action_jax)
CTRL_RANGE = jnp.array(mj_model.actuator_ctrlrange)

print(f"Root body ({_cfg['root_body']}) id: {ROOT_BODY_ID}")
print(f"Action dim: {mj_model.nu}")

In [ ]:
def get_obs(data):
    """Extract observation vector from MJX data using shared obs builder."""
    return build_bipedal_obs(
        qpos=data.qpos,
        qvel=data.qvel,
        sensordata=data.sensordata,
        pelvis_xpos=data.xpos[ROOT_BODY_ID],
        target_pos=jnp.zeros(3),  # Target tracking handled by reward config
        sensor_layout=SENSOR_LAYOUT,
    )


# ---------- Pure-JAX reward & termination (vmap-safe) ----------
# These now use the shared reward_functions which are JAX-trace-safe.

def compute_reward(data, action, reward_cfg):
    """Compute scalar reward using shared JAX-trace-safe reward functions."""
    vel_2d = data.qvel[:2]
    forward_dir = jnp.array([1.0, 0.0])

    # Forward velocity reward
    r_forward, _fwd_vel = reward_forward_velocity(
        vel_2d, forward_dir, 8.0, reward_cfg["forward_vel_weight"]
    )

    # Alive bonus
    r_alive = reward_alive(reward_cfg["alive_bonus"])

    # Energy penalty
    r_energy = reward_energy(action, CTRL_RANGE.shape[0], reward_cfg["energy_penalty_weight"])

    # Posture reward (quadratic tilt penalty)
    root_quat = data.sensordata[6:10]
    r_posture, _tilt = reward_posture(root_quat, MAX_TILT_ANGLE, reward_cfg.get("posture_weight", 0.2))

    return r_forward + r_alive + r_energy + r_posture


def is_terminated(data):
    """Check if the dinosaur has fallen using shared termination check."""
    body_z = data.xpos[ROOT_BODY_ID, 2]
    root_quat = data.sensordata[6:10]
    tilt = quat_to_tilt(root_quat)
    terminated, _ = check_height_tilt_termination(
        body_z, tilt, (HEALTHY_Z_MIN, HEALTHY_Z_MAX), MAX_TILT_ANGLE
    )
    return terminated


def scale_action(action):
    """Scale action from [-1, 1] to actuator control range using shared function."""
    return scale_action_jax(action, CTRL_RANGE)


# Verify obs dimension
mujoco.mj_forward(mj_model, mj_data)
_test_data = mjx.put_data(mj_model, mj_data)
_test_obs = get_obs(_test_data)
OBS_DIM = _test_obs.shape[0]
ACT_DIM = mj_model.nu
print(f"Observation dim: {OBS_DIM}")
print(f"Action dim: {ACT_DIM}")

## 3. Batched MJX Step

A single `jax.jit`-compiled function that steps `N` parallel environments.

In [ ]:
def mjx_step_single(model, data, action):
    """Step one environment: apply action, advance physics, return new data."""
    ctrl = scale_action(action)
    data = data.replace(ctrl=ctrl)

    # Frame skip: step physics multiple times per action
    def body_fn(_, d):
        return mjx.step(model, d)

    data = jax.lax.fori_loop(0, FRAME_SKIP, body_fn, data)
    return data


# Vectorize: model is shared (None), data and action are batched (0)
@jax.jit
def batched_step(model, data_batch, action_batch):
    return jax.vmap(mjx_step_single, in_axes=(None, 0, 0))(model, data_batch, action_batch)


print("Batched step function compiled.")

## 4. Policy Network (Flax)
Uses the shared `ActorCritic` from `environments.shared.jax_ppo`.

In [ ]:
# Initialize using the shared ActorCritic from jax_ppo
network = make_actor_critic(action_dim=ACT_DIM)
rng = jax.random.PRNGKey(42)
dummy_obs = jnp.zeros((OBS_DIM,))
params = network.init(rng, dummy_obs)

# Observation normalization (stabilizes training across species/stages)
obs_rms = RunningMeanStd.create(OBS_DIM)

# Resume from checkpoint if specified
_resume_update = 0
_jax_kw_resume = {}
try:
    from environments.shared.config import load_stage_config as _lsc
    _jax_kw_resume = _lsc(SPECIES, CURRENT_STAGE).get("jax_kwargs", {})
except Exception:
    pass

if RESUME_FROM:
    import pickle
    _ckpt_path = OUTPUT_DIR / RESUME_FROM if not Path(RESUME_FROM).is_absolute() else Path(RESUME_FROM)
    with open(_ckpt_path, "rb") as f:
        _ckpt = pickle.load(f)
    params = jax.device_put(_ckpt["params"])
    _resume_update = _ckpt.get("update", 0)
    if "obs_rms" in _ckpt:
        obs_rms = _ckpt["obs_rms"]
        # When transitioning between stages, the obs distribution shifts
        # (e.g. near-zero velocity in balance → sustained velocity in
        # locomotion).  With 2048 envs the prior count can be ~65M, making
        # update_running_stats nearly a no-op.  Decay the count so new
        # data adapts the statistics within a few updates.
        _obs_decay = _jax_kw_resume.get("obs_rms_decay_on_resume", 0.01)
        if _obs_decay < 1.0:
            _old_count = obs_rms.count
            obs_rms = decay_running_stats(obs_rms, decay_factor=_obs_decay)
            print(f"  obs_rms count decayed: {_old_count:,.0f} → {obs_rms.count:,.0f} (factor={_obs_decay})")
    print(f"Resumed from {_ckpt_path} (update {_resume_update})")
    if "reward_history" in _ckpt:
        print(f"  Prior history: {len(_ckpt['reward_history'])} updates, best reward: {max(_ckpt['reward_history']):.4f}")

n_params = sum(p.size for p in jax.tree.leaves(params))
print(f"ActorCritic parameters: {n_params:,}")

## 5. PPO Implementation
Core PPO functions (`sample_action`, `compute_gae`, `ppo_loss`) are now
imported from `environments.shared.jax_ppo`. Notebook-specific wrappers
below adapt them for the training loop.

In [ ]:
# Use shared sample_action but with our network closure
def nb_sample_action(params, obs, rng):
    """Sample action from Gaussian policy using shared PPO module."""
    return sample_action(params, network, obs, rng)


# Use shared compute_gae directly (same signature)
# compute_gae is already imported from jax_ppo

# Use shared ppo_loss directly — it now includes clip_fraction and mean_std
# in the returned info dict. Create a notebook wrapper that adapts the
# interface for the training loop (positional args -> batch dict).
def nb_ppo_loss(params, obs, actions, old_log_probs, advantages, returns,
                clip_range=0.2, vf_coef=0.5, ent_coef=0.01):
    """PPO loss using shared implementation with full diagnostics."""
    batch = {
        "obs": obs,
        "action": actions,
        "old_log_prob": old_log_probs,
        "advantage": advantages,
        "return_": returns,
    }
    config = PPOConfig(
        clip_range=clip_range,
        vf_coef=vf_coef,
        ent_coef=ent_coef,
    )
    return ppo_loss(params, network, batch, config)


print("PPO functions defined (using shared modules from jax_ppo).")

## 6. Training Loop

In [ ]:
# ---------- Hyperparameters (loaded from TOML [jax] section) ----------
from environments.shared.config import load_stage_config

_stage_cfg = load_stage_config(SPECIES, CURRENT_STAGE)
_env_kw = _stage_cfg["env_kwargs"]
_jax_kw = _stage_cfg.get("jax_kwargs", {})

# JAX/MJX training hyperparameters (from TOML, with notebook defaults as fallback)
NUM_ENVS = _jax_kw.get("num_envs", 2048)
ROLLOUT_LEN = _jax_kw.get("rollout_len", 64)
NUM_UPDATES = _jax_kw.get("num_updates", 500)
PPO_EPOCHS = _jax_kw.get("ppo_epochs", 4)
MINIBATCH_SIZE = _jax_kw.get("minibatch_size", 512)
LEARNING_RATE = _jax_kw.get("learning_rate", 3e-4)
MAX_GRAD_NORM = _jax_kw.get("max_grad_norm", 0.5)
GAMMA = _jax_kw.get("gamma", 0.99)
GAE_LAMBDA = _jax_kw.get("gae_lambda", 0.95)
CLIP_RANGE = _jax_kw.get("clip_range", 0.2)
ENT_COEF = _jax_kw.get("ent_coef", 0.01)
FALL_PENALTY = _jax_kw.get("fall_penalty", -10.0)
RESET_NOISE_SCALE = _jax_kw.get("reset_noise_scale", 0.05)
INIT_QPOS_NOISE = _jax_kw.get("init_qpos_noise", 0.01)
INIT_YAW_NOISE = _jax_kw.get("init_yaw_noise", 0.1)

# Curriculum warmup (constrains policy updates while critic adapts to new reward landscape)
WARMUP_UPDATES = _jax_kw.get("warmup_updates", 0)          # 0 = no warmup (Stage 1 default)
WARMUP_CLIP_RANGE = _jax_kw.get("warmup_clip_range", 0.02)
WARMUP_ENT_COEF = _jax_kw.get("warmup_ent_coef", 0.02)

# Reward ramp (linearly ramp a reward weight from a fraction to its full value)
RAMP_UPDATES = _jax_kw.get("ramp_updates", 0)              # 0 = no ramp (Stage 1 default)
RAMP_ATTR = _jax_kw.get("ramp_attr", "forward_vel_weight")
RAMP_START_FRACTION = _jax_kw.get("ramp_start_fraction", 0.1)

# Reward config from [env] section
reward_cfg = {
    "forward_vel_weight": _env_kw.get("forward_vel_weight", 0.0),
    "alive_bonus": _env_kw.get("alive_bonus", 1.0),
    "energy_penalty_weight": _env_kw.get("energy_penalty_weight", 0.001),
    "posture_weight": _env_kw.get("posture_weight", 0.2),
}

print("Training config (from TOML [jax] section):")
print(f"  Species: {SPECIES}")
print(f"  Envs: {NUM_ENVS}")
print(f"  Rollout length: {ROLLOUT_LEN}")
print(f"  Updates: {NUM_UPDATES}")
print(f"  Total env steps: {NUM_ENVS * ROLLOUT_LEN * NUM_UPDATES:,}")
print(f"  Stage: {CURRENT_STAGE} ({_cfg['stage_names'].get(CURRENT_STAGE, '?')})")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Gamma: {GAMMA}")
print(f"  Clip range: {CLIP_RANGE}")
print(f"  Entropy coef: {ENT_COEF}")
print(f"  Max grad norm: {MAX_GRAD_NORM}")
print(f"  Fall penalty: {FALL_PENALTY}")
print(f"  Reset noise: joints={RESET_NOISE_SCALE}, xy={INIT_QPOS_NOISE}, yaw={INIT_YAW_NOISE}")
if WARMUP_UPDATES > 0:
    print(f"  Warmup: {WARMUP_UPDATES} updates (clip={WARMUP_CLIP_RANGE}, ent={WARMUP_ENT_COEF})")
if RAMP_UPDATES > 0:
    print(f"  Reward ramp: {RAMP_ATTR} from {RAMP_START_FRACTION:.0%} to 100% over {RAMP_UPDATES} updates")
print(f"  Reward config: {reward_cfg}")

In [ ]:
# Initialize batched environments
rng = jax.random.PRNGKey(42)

# Reset: create initial MJX data for all envs
mujoco.mj_resetData(mj_model, mj_data)
mujoco.mj_forward(mj_model, mj_data)
base_data = mjx.put_data(mj_model, mj_data)


# Replicate across batch with perturbations to position, orientation, and joints
def init_env(rng):
    rng_joint, rng_xy, rng_yaw = jax.random.split(rng, 3)

    # Joint angle perturbation
    joint_noise = jax.random.uniform(
        rng_joint, (mj_model.nq - 7,),
        minval=-RESET_NOISE_SCALE, maxval=RESET_NOISE_SCALE,
    )

    # XY position jitter (root position: qpos[0:2])
    xy_noise = jax.random.uniform(rng_xy, (2,), minval=-INIT_QPOS_NOISE, maxval=INIT_QPOS_NOISE)

    # Yaw rotation perturbation (rotate root quaternion around Z axis)
    yaw_angle = jax.random.uniform(rng_yaw, (), minval=-INIT_YAW_NOISE, maxval=INIT_YAW_NOISE)
    half_yaw = yaw_angle / 2.0
    # Yaw quaternion: [cos(yaw/2), 0, 0, sin(yaw/2)]
    yaw_quat = jnp.array([jnp.cos(half_yaw), 0.0, 0.0, jnp.sin(half_yaw)])
    # Multiply: yaw_quat * base_quat (Hamilton product)
    base_quat = base_data.qpos[3:7]
    w1, x1, y1, z1 = yaw_quat
    w2, x2, y2, z2 = base_quat
    new_quat = jnp.array([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2,
    ])

    qpos = base_data.qpos
    qpos = qpos.at[0:2].add(xy_noise)
    qpos = qpos.at[3:7].set(new_quat)
    qpos = qpos.at[7:].add(joint_noise)

    return base_data.replace(qpos=qpos)


rngs = jax.random.split(rng, NUM_ENVS)
env_batch = jax.vmap(init_env)(rngs)

# Forward pass to update derived quantities
env_batch = jax.jit(jax.vmap(mjx.forward, in_axes=(None, 0)))(mjx_model, env_batch)

print(f"Initialized {NUM_ENVS} parallel environments.")
print(f"qpos batch shape: {env_batch.qpos.shape}")
print(f"Reset noise: joints=±{RESET_NOISE_SCALE}, xy=±{INIT_QPOS_NOISE}m, yaw=±{INIT_YAW_NOISE}rad")

In [ ]:
# Optimizer (with gradient clipping)
optimizer = make_optimizer(PPOConfig(learning_rate=LEARNING_RATE, max_grad_norm=MAX_GRAD_NORM))
opt_state = optimizer.init(params)


# JIT-compiled PPO update step (with gradient norm and loss decomposition)
@jax.jit
def ppo_update(params, opt_state, obs, actions, log_probs, advantages, returns,
               clip_range=CLIP_RANGE, ent_coef=ENT_COEF):
    (loss, aux), grads = jax.value_and_grad(nb_ppo_loss, has_aux=True)(
        params,
        obs,
        actions,
        log_probs,
        advantages,
        returns,
        clip_range=clip_range,
        ent_coef=ent_coef,
    )
    grad_norm = optax.global_norm(grads)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss, aux, grad_norm


# JIT-compiled batched action sampling
@jax.jit
def batched_sample(params, obs_batch, rng):
    rngs = jax.random.split(rng, obs_batch.shape[0])
    return jax.vmap(nb_sample_action, in_axes=(None, 0, 0))(params, obs_batch, rngs)


# JIT-compiled batched observation + reward
@jax.jit
def batched_obs(data_batch):
    return jax.vmap(get_obs)(data_batch)


@jax.jit
def batched_reward(data_batch, action_batch):
    return jax.vmap(compute_reward, in_axes=(0, 0, None))(data_batch, action_batch, reward_cfg)


@jax.jit
def batched_terminated(data_batch):
    return jax.vmap(is_terminated)(data_batch)


# Reset helper: reset fallen envs using a pre-computed fresh batch
@jax.jit
def reset_fallen(env_batch, dones, fresh_batch):
    """Reset environments where done=True using pre-computed fresh states."""
    def select(fresh_field, existing_field):
        expand = dones.reshape((-1,) + (1,) * (existing_field.ndim - 1))
        return jnp.where(expand, fresh_field, existing_field)

    return jax.tree.map(select, fresh_batch, env_batch)


# Pre-compute fresh batch helper (jitted for speed)
@jax.jit
def make_fresh_batch(rng):
    """Create a fresh batch of environments for resetting fallen envs."""
    rngs = jax.random.split(rng, NUM_ENVS)
    fresh = jax.vmap(init_env)(rngs)
    return jax.jit(jax.vmap(mjx.forward, in_axes=(None, 0)))(mjx_model, fresh)


# ==========================================================================
# OPTIMISED: Scanned PPO update epochs
# ==========================================================================
# Replaces the Python double-loop (PPO_EPOCHS x n_minibatches) with nested
# jax.lax.scan, compiling the entire PPO update into one XLA program.

@jax.jit
def scan_ppo_epochs(params, opt_state, flat_obs, flat_act, flat_lp, flat_adv,
                    flat_ret, rng, clip_range, ent_coef):
    """Run PPO_EPOCHS of minibatch gradient updates, fully compiled."""
    total_samples = flat_obs.shape[0]
    n_minibatches = total_samples // MINIBATCH_SIZE

    def epoch_fn(carry, _):
        params, opt_state, rng = carry
        rng, rng_perm = jax.random.split(rng)
        perm = jax.random.permutation(rng_perm, total_samples)

        def to_mbs(arr):
            return arr[perm[:n_minibatches * MINIBATCH_SIZE]].reshape(
                n_minibatches, MINIBATCH_SIZE, *arr.shape[1:])

        mb_data = (to_mbs(flat_obs), to_mbs(flat_act), to_mbs(flat_lp),
                   to_mbs(flat_adv), to_mbs(flat_ret))

        def mb_step(carry, mb):
            params, opt_state = carry
            obs, act, lp, adv, ret = mb
            params, opt_state, loss, aux, gn = ppo_update(
                params, opt_state, obs, act, lp, adv, ret,
                clip_range=clip_range, ent_coef=ent_coef)
            return (params, opt_state), (loss, aux, gn)

        (params, opt_state), (losses, auxs, gns) = jax.lax.scan(
            mb_step, (params, opt_state), mb_data)
        return (params, opt_state, rng), (losses, auxs, gns)

    (params, opt_state, _), (all_losses, all_auxs, all_gns) = jax.lax.scan(
        epoch_fn, (params, opt_state, rng), None, length=PPO_EPOCHS)

    mean_loss = jnp.mean(all_losses)
    mean_gn = jnp.mean(all_gns)
    mean_aux = jax.tree.map(jnp.mean, all_auxs)
    return params, opt_state, mean_loss, mean_aux, mean_gn


print("Training functions compiled (using shared optax optimizer).")
print("  rollout: Python loop with async GPU dispatch (physics-bound)")
print("  [optimised] scan_ppo_epochs: fused PPO updates via jax.lax.scan")


In [ ]:
# ---------- Main Training Loop ----------
# Rollout: Python loop with async GPU dispatch (physics is the bottleneck,
# not dispatch overhead — scan added 281s compile time for 0% speedup).
# PPO updates: jax.lax.scan (compiles fast, gives real speedup).

from environments.shared.jax_checkpoint import CheckpointManager, save_checkpoint
from environments.shared.jax_training_utils import (
    StabilityMonitor,
    TrainingCSVLogger,
    RolloutProfiler,
    compute_episode_stats,
)
from environments.shared.reporting import format_duration

CHECKPOINT_FREQ = 25  # Save checkpoint every N updates
MAX_CHECKPOINTS = 5   # Keep only the last N checkpoints (saves disk space)

# Console log frequency based on VERBOSE:
# 0 = only final summary, 1 = every 20 updates (default), 2 = every update
_LOG_INTERVAL = {0: None, 1: 20, 2: 1}.get(VERBOSE, 20)

reward_history = []
loss_history = []
diagnostics_history = []
episode_return_history = []

# Best model tracking
best_reward = -float("inf")
best_params = None
best_update = -1

# Episode return accumulators (on-device — no host syncs during rollout)
_ep_returns = jnp.zeros(NUM_ENVS)
_ep_lengths = jnp.zeros(NUM_ENVS, dtype=jnp.int32)

# --- Library utilities (replace inline implementations) ---
_ckpt_mgr = CheckpointManager(MODEL_DIR, prefix=f"{SPECIES}_jax_checkpoint", max_keep=MAX_CHECKPOINTS)
_stability = StabilityMonitor()
_profiler = RolloutProfiler(interval=50)
_csv_logger = TrainingCSVLogger(OUTPUT_DIR / f"{SPECIES}_jax_training_log.csv")
csv_path = _csv_logger.path  # Expose for downstream cells

# Warmup/ramp state
_warmup_active = WARMUP_UPDATES > 0
_ramp_active = RAMP_UPDATES > 0
_ramp_target_value = reward_cfg.get(RAMP_ATTR, 0.0) if _ramp_active else 0.0

if _warmup_active:
    _original_clip_range = CLIP_RANGE
    _original_ent_coef = ENT_COEF

# ==================== GPU / Device Diagnostics ====================
_device = jax.devices()[0]
_dev_kind = _device.device_kind if hasattr(_device, 'device_kind') else str(_device.platform).upper()
print(f"Device:     {_device} ({_dev_kind})")
print(f"JAX:        {jax.__version__}")
try:
    _mem_stats = _device.memory_stats()
    if _mem_stats:
        _total_gb = _mem_stats.get('bytes_limit', 0) / 1e9
        _used_gb  = _mem_stats.get('peak_bytes_in_use', _mem_stats.get('bytes_in_use', 0)) / 1e9
        print(f"GPU memory: {_used_gb:.1f} / {_total_gb:.1f} GB used")
except Exception:
    print("GPU memory: (stats unavailable)")
_batch_size_total = ROLLOUT_LEN * NUM_ENVS
print(f"Batch size: {_batch_size_total:,} ({ROLLOUT_LEN} steps x {NUM_ENVS} envs)")
print(f"PPO:        {PPO_EPOCHS} epochs x {_batch_size_total // MINIBATCH_SIZE} minibatches of {MINIBATCH_SIZE}")
_total_env_steps = NUM_UPDATES * ROLLOUT_LEN * NUM_ENVS
print(f"Total:      {_total_env_steps:,} env steps over {NUM_UPDATES} updates")

_start_update = _resume_update
print(f"\nStarting training: updates {_start_update}..{_start_update + NUM_UPDATES - 1} "
      f"({ROLLOUT_LEN} steps x {NUM_ENVS} envs)")
print(f"Checkpoint frequency: every {CHECKPOINT_FREQ} updates (keep last {MAX_CHECKPOINTS})")
if _warmup_active:
    print(f"Warmup: updates 0..{WARMUP_UPDATES - 1} (clip_range={WARMUP_CLIP_RANGE}, ent_coef={WARMUP_ENT_COEF})")
if _ramp_active:
    print(f"Reward ramp: {RAMP_ATTR} from {_ramp_target_value * RAMP_START_FRACTION:.4f} to {_ramp_target_value:.4f} over updates 0..{RAMP_UPDATES - 1}")
print(f"CSV log: {csv_path}")
print("=" * 70)

t_start = time.time()
_cum_t_rollout = 0.0
_cum_t_ppo = 0.0
_compile_time = 0.0

try:
    for update in range(_start_update, _start_update + NUM_UPDATES):
        # ---------- Pre-compute fresh batch for resets ----------
        rng, rng_fresh = jax.random.split(rng)
        fresh_batch = make_fresh_batch(rng_fresh)

        # ---------- Warmup ----------
        relative_update = update - _start_update
        if _warmup_active:
            if relative_update < WARMUP_UPDATES:
                _active_clip_range = WARMUP_CLIP_RANGE
                _active_ent_coef = WARMUP_ENT_COEF
            else:
                _active_clip_range = _original_clip_range
                _active_ent_coef = _original_ent_coef
                if relative_update == WARMUP_UPDATES and (_LOG_INTERVAL is not None):
                    print(f"  >>> Warmup complete at update {update}: restoring clip_range={_original_clip_range}, ent_coef={_original_ent_coef}")
        else:
            _active_clip_range = CLIP_RANGE
            _active_ent_coef = ENT_COEF

        # ---------- Reward ramp ----------
        if _ramp_active:
            if relative_update < RAMP_UPDATES:
                ramp_progress = relative_update / RAMP_UPDATES
                ramp_value = _ramp_target_value * (RAMP_START_FRACTION + (1.0 - RAMP_START_FRACTION) * ramp_progress)
            else:
                ramp_value = _ramp_target_value
            reward_cfg[RAMP_ATTR] = ramp_value

        # ---------- Collect rollout (Python loop, async GPU dispatch) ----------
        _t_phase = time.time()
        all_obs, all_actions, all_log_probs, all_values = [], [], [], []
        all_rewards, all_dones = [], []

        _do_profile = _profiler.should_profile(relative_update)
        _op_times = {"obs": 0, "policy": 0, "physics": 0, "reward": 0, "reset": 0}
        _contact_counts = []

        for t in range(ROLLOUT_LEN):
            rng, rng_act = jax.random.split(rng)

            if _do_profile: _t0 = time.time()
            obs_raw = batched_obs(env_batch)
            obs = normalize_obs(obs_raw, obs_rms)
            if _do_profile:
                jax.block_until_ready(obs)
                _op_times["obs"] += time.time() - _t0
                _t0 = time.time()

            actions, log_probs, values = batched_sample(params, obs, rng_act)
            if _do_profile:
                jax.block_until_ready(actions)
                _op_times["policy"] += time.time() - _t0
                _t0 = time.time()

            # Step physics (async GPU dispatch)
            env_batch = batched_step(mjx_model, env_batch, actions)
            if _do_profile:
                jax.block_until_ready(env_batch.qpos)
                _op_times["physics"] += time.time() - _t0
                _t0 = time.time()

            rewards = batched_reward(env_batch, actions)
            terminated = batched_terminated(env_batch)
            if _do_profile:
                jax.block_until_ready(terminated)
                _op_times["reward"] += time.time() - _t0

            # Track active contacts (sample first env, lightweight)
            if _do_profile and t == 0:
                try:
                    _ncon = int(env_batch.ncon[0]) if hasattr(env_batch, 'ncon') else -1
                    _contact_counts.append(_ncon)
                except Exception:
                    pass

            # Episode tracking (fully on-device, no host syncs)
            _ep_lengths = _ep_lengths + 1
            truncated = _ep_lengths >= MAX_EPISODE_STEPS
            dones = terminated | truncated

            rewards = rewards + terminated.astype(jnp.float32) * FALL_PENALTY
            _ep_returns = _ep_returns + rewards

            all_obs.append(obs)
            all_actions.append(actions)
            all_log_probs.append(log_probs)
            all_values.append(values)
            all_rewards.append(rewards)
            all_dones.append(dones.astype(jnp.float32))

            # Reset done envs (on-device)
            if _do_profile: _t0 = time.time()
            env_batch = reset_fallen(env_batch, dones, fresh_batch)
            _ep_returns = jnp.where(dones, 0.0, _ep_returns)
            _ep_lengths = jnp.where(dones, 0, _ep_lengths)
            if _do_profile:
                jax.block_until_ready(env_batch.qpos)
                _op_times["reset"] += time.time() - _t0

        # Stack rollout data
        obs_t = jnp.stack(all_obs)
        act_t = jnp.stack(all_actions)
        lp_t = jnp.stack(all_log_probs)
        val_t = jnp.stack(all_values)
        rew_t = jnp.stack(all_rewards)
        done_t = jnp.stack(all_dones)

        # Block for accurate timing
        jax.block_until_ready(done_t)
        _t_rollout = time.time() - _t_phase
        _cum_t_rollout += _t_rollout

        # Print per-operation breakdown (when profiling is active)
        if _do_profile:
            _profiler.record(update, _op_times)
            if _LOG_INTERVAL is not None:
                _total_op = sum(_op_times.values())
                print(f"  [profile update {update}] rollout={_t_rollout:.1f}s breakdown:")
                for _opname, _optime in _op_times.items():
                    _pct = 100 * _optime / _total_op if _total_op > 0 else 0
                    print(f"    {_opname:8s}: {_optime:6.2f}s ({_pct:4.1f}%)")
                if _contact_counts and _contact_counts[0] >= 0:
                    print(f"    contacts: {_contact_counts[0]} active (step 0)")

        # ---------- Episode stats (using library helper) ----------
        done_t_np = np.array(done_t)
        rew_np = np.array(rew_t)
        fall_rate = float(done_t_np.sum()) / (ROLLOUT_LEN * NUM_ENVS)
        _completed_returns, _completed_lengths = compute_episode_stats(rew_np, done_t_np)

        # ---------- Update obs normalisation (once per update) ----------
        obs_batch_flat = obs_t.reshape(-1, OBS_DIM)
        obs_rms = update_running_stats(obs_rms, obs_batch_flat)

        # ---------- Bootstrap value for GAE ----------
        rng, rng_bootstrap = jax.random.split(rng)
        obs_final_raw = batched_obs(env_batch)
        obs_final = normalize_obs(obs_final_raw, obs_rms)
        _, _, bootstrap_values = batched_sample(params, obs_final, rng_bootstrap)
        val_t_plus1 = jnp.concatenate([val_t, bootstrap_values[None]], axis=0)

        # ---------- Compute advantages ----------
        advantages, returns = compute_gae(rew_t, val_t_plus1, done_t, GAMMA, GAE_LAMBDA)

        # Flatten: (T * N, ...)
        flat_obs = obs_t.reshape(-1, OBS_DIM)
        flat_act = act_t.reshape(-1, ACT_DIM)
        flat_lp = lp_t.reshape(-1)
        flat_adv = advantages.reshape(-1)
        flat_ret = returns.reshape(-1)

        # ---------- PPO update (fused scan — compiles fast, real speedup) ----------
        _t_phase = time.time()
        rng, rng_ppo = jax.random.split(rng)
        params, opt_state, avg_loss, avg_aux, avg_grad_norm = scan_ppo_epochs(
            params, opt_state, flat_obs, flat_act, flat_lp, flat_adv, flat_ret,
            rng_ppo, jnp.float32(_active_clip_range), jnp.float32(_active_ent_coef),
        )

        # Transfer scalar metrics to host
        avg_loss = float(avg_loss)
        avg_grad_norm = float(avg_grad_norm)
        avg_aux = {k: float(v) for k, v in avg_aux.items()}
        _t_ppo = time.time() - _t_phase
        _cum_t_ppo += _t_ppo

        avg_reward = float(rew_t.mean())

        # Episode return stats
        if _completed_returns:
            mean_ep_return = np.mean(_completed_returns)
            mean_ep_length = np.mean(_completed_lengths)
        else:
            mean_ep_return = float("nan")
            mean_ep_length = float("nan")
        episode_return_history.append(mean_ep_return)

        reward_history.append(avg_reward)
        loss_history.append(avg_loss)
        diagnostics_history.append({
            "reward": avg_reward,
            "episode_return": mean_ep_return,
            "episode_length": mean_ep_length,
            "loss": avg_loss,
            "grad_norm": avg_grad_norm,
            "fall_rate": fall_rate,
            "t_rollout": _t_rollout,
            "t_ppo": _t_ppo,
            **avg_aux,
        })

        # ==================== Stability watchdog ====================
        _kl = avg_aux.get("approx_kl", 0.0)
        should_halt, _is_unstable, _stab_msg = _stability.check(
            _kl, avg_grad_norm, avg_loss, update
        )
        if _stab_msg:
            print(f"  {_stab_msg}")
        if should_halt:
            break

        # Track best model
        _track_metric = mean_ep_return if not np.isnan(mean_ep_return) else avg_reward
        if _track_metric > best_reward and not _is_unstable:
            best_reward = _track_metric
            best_params = jax.device_get(params)
            best_update = update

        # CSV log
        elapsed = time.time() - t_start
        steps_done = (update - _start_update + 1) * ROLLOUT_LEN * NUM_ENVS
        sps = steps_done / elapsed
        _csv_logger.log({
            "update": update,
            "reward_per_step": f"{avg_reward:.4f}",
            "episode_return": f"{mean_ep_return:.2f}" if not np.isnan(mean_ep_return) else "",
            "episode_length": f"{mean_ep_length:.1f}" if not np.isnan(mean_ep_length) else "",
            "total_loss": f"{avg_loss:.4f}",
            "policy_loss": f"{avg_aux['policy_loss']:.4f}",
            "value_loss": f"{avg_aux['value_loss']:.4f}",
            "entropy": f"{avg_aux['entropy']:.4f}",
            "approx_kl": f"{avg_aux['approx_kl']:.6f}",
            "clip_fraction": f"{avg_aux['clip_fraction']:.4f}",
            "grad_norm": f"{avg_grad_norm:.4f}",
            "mean_std": f"{avg_aux['mean_std']:.4f}",
            "steps": steps_done,
            "sps": f"{sps:.0f}",
            "fall_rate": f"{fall_rate:.4f}",
            "elapsed": f"{elapsed:.1f}",
            "t_rollout": f"{_t_rollout:.3f}",
            "t_ppo": f"{_t_ppo:.3f}",
        })

        # Console logging
        if _LOG_INTERVAL is not None and (
            (update - _start_update) % _LOG_INTERVAL == 0
            or update == _start_update + NUM_UPDATES - 1
        ):
            updates_done = update - _start_update + 1
            updates_left = NUM_UPDATES - updates_done
            eta = (elapsed / updates_done) * updates_left if updates_done > 0 else 0
            eta_str = f"{eta / 60:.0f}m" if eta > 60 else f"{eta:.0f}s"
            ep_ret_str = f"ep_ret={mean_ep_return:+.1f}" if not np.isnan(mean_ep_return) else "ep_ret=n/a"
            print(
                f"[{update:4d}/{_start_update + NUM_UPDATES}]  "
                f"r/step={avg_reward:+.3f}  {ep_ret_str}  "
                f"loss={avg_loss:.4f}  "
                f"pi={avg_aux['policy_loss']:.3f}  v={avg_aux['value_loss']:.3f}  "
                f"ent={avg_aux['entropy']:.3f}  kl={avg_aux['approx_kl']:.4f}  "
                f"grad={avg_grad_norm:.3f}  falls={fall_rate:.1%}  "
                f"SPS={sps:,.0f}  ETA={eta_str}  "
                f"[{_t_rollout:.1f}s+{_t_ppo:.2f}s]"
            )

        # Periodic checkpointing (with rotation via CheckpointManager)
        if (update - _start_update + 1) % CHECKPOINT_FREQ == 0:
            _ckpt_mgr.save(
                params, update + 1, obs_rms=obs_rms,
                history={
                    "reward": reward_history,
                    "loss": loss_history,
                    "episode_return": episode_return_history,
                },
            )
            if _LOG_INTERVAL is not None:
                print(f"  >>> Checkpoint saved: {_ckpt_mgr.latest}")

finally:
    _csv_logger.close()

# ==================== Training Summary ====================
elapsed = time.time() - t_start
total_steps = NUM_UPDATES * ROLLOUT_LEN * NUM_ENVS
print("=" * 70)
print(f"Done! {total_steps:,} steps in {format_duration(elapsed)} ({total_steps / elapsed:,.0f} SPS)")
print(f"Best metric: {best_reward:+.4f} at update {best_update}")

# Timing breakdown
_pct_rollout = 100 * _cum_t_rollout / elapsed if elapsed > 0 else 0
_pct_ppo     = 100 * _cum_t_ppo / elapsed if elapsed > 0 else 0
_pct_other   = 100 - _pct_rollout - _pct_ppo
print(f"\nTiming breakdown:")
print(f"  Rollout (MJX physics): {_cum_t_rollout:7.1f}s  ({_pct_rollout:4.1f}%)")
print(f"  PPO updates:           {_cum_t_ppo:7.1f}s  ({_pct_ppo:4.1f}%)")
print(f"  Other (GAE/IO/log):    {elapsed - _cum_t_rollout - _cum_t_ppo:7.1f}s  ({_pct_other:4.1f}%)")

# Physics throughput
_physics_steps = NUM_UPDATES * ROLLOUT_LEN * NUM_ENVS * FRAME_SKIP
if _cum_t_rollout > 0:
    print(f"\nMJX physics: {_physics_steps:,} steps in {_cum_t_rollout:.1f}s "
          f"({_physics_steps / _cum_t_rollout:,.0f} physics steps/sec)")

# Reward trend
if len(reward_history) >= 10:
    _first10 = np.mean(reward_history[:10])
    _last10 = np.mean(reward_history[-10:])
    _trend = "improved" if _last10 > _first10 else "declined"
    print(f"\nReward trend: {_first10:.3f} (first 10) -> {_last10:.3f} (last 10)  [{_trend}]")

if _stability.total_warnings > 0:
    print(f"\nTotal stability warnings: {_stability.total_warnings}")

# Per-operation profiling summary
_prof_summary = _profiler.summary()
if _prof_summary:
    print(f"\n{_prof_summary}")

# GPU memory after training
try:
    _mem_stats = _device.memory_stats()
    if _mem_stats:
        _peak_gb = _mem_stats.get('peak_bytes_in_use', 0) / 1e9
        print(f"Peak GPU memory: {_peak_gb:.2f} GB")
except Exception:
    pass

# Save final trained parameters
params_path = MODEL_DIR / f"{SPECIES}_jax_params.pkl"
save_checkpoint(
    params_path, params, obs_rms=obs_rms,
    extra={
        "best_params": best_params,
        "best_reward": best_reward,
        "best_update": best_update,
    },
    history={
        "reward": reward_history,
        "loss": loss_history,
        "episode_return": episode_return_history,
        "diagnostics": diagnostics_history,
    },
)
print(f"\nParameters and training history saved to: {params_path}")
print(f"Training log CSV saved to: {csv_path}")

## Stage Gate Evaluation

Run full evaluation episodes on CPU to check whether the curriculum gate
thresholds (min reward and min episode length) have been met. Both conditions
must pass before advancing to the next training stage.

In [ ]:
# ---------- Stage Gate Evaluation ----------
# Uses shared evaluate_policy_cpu and check_stage_gate from jax_eval.

from environments.shared.config import load_stage_config

# Load gate thresholds from TOML config
stage_config = load_stage_config(SPECIES, CURRENT_STAGE)
curriculum = stage_config.get("curriculum_kwargs", {})
gate_min_reward = curriculum.get("min_avg_reward", -float("inf"))
gate_min_length = curriculum.get("min_avg_episode_length", 0)

print(f"Stage {CURRENT_STAGE} curriculum gate thresholds:")
print(f"  min_avg_reward:         {gate_min_reward}")
print(f"  min_avg_episode_length: {gate_min_length}")

# Use best params from training
eval_params = best_params if best_params is not None else jax.device_get(params)

# Configure evaluation
eval_config = EvalConfig(
    n_episodes=30,
    max_episode_steps=MAX_EPISODE_STEPS,
    frame_skip=FRAME_SKIP,
    healthy_z_range=(HEALTHY_Z_MIN, HEALTHY_Z_MAX),
    max_tilt_angle=MAX_TILT_ANGLE,
    root_body_id=ROOT_BODY_ID,
    sensor_quat_start=6,
    reset_noise_scale=0.01,
    forward_vel_max=8.0,
)

# Foot sensor indices for diagnostics
_foot_indices = tuple(
    i for i in SENSOR_LAYOUT.foot_indices
) if hasattr(SENSOR_LAYOUT, "foot_indices") else ()

print(f"\nRunning {eval_config.n_episodes} evaluation episodes on CPU...")

eval_results = evaluate_policy_cpu(
    mj_model, eval_params, network, obs_rms,
    get_obs_fn=get_obs,
    normalize_obs_fn=normalize_obs,
    scale_action_fn=scale_action,
    reward_fn=compute_reward,
    reward_cfg=reward_cfg,
    config=eval_config,
    foot_sensor_indices=_foot_indices,
)

# Unpack results for backward compat with downstream cells
eval_rewards = eval_results.rewards
eval_lengths = eval_results.lengths
mean_reward = eval_results.mean_reward
std_reward = eval_results.std_reward
mean_length = eval_results.mean_length
std_length = eval_results.std_length
mean_fwd_vel = eval_results.mean_forward_vel
std_fwd_vel = float(np.std(eval_results.forward_vels)) if eval_results.forward_vels else 0.0
mean_distance = eval_results.mean_distance
mean_tilt = eval_results.mean_tilt
mean_height = eval_results.mean_height

# Also expose diag_ variables for any downstream cells that reference them
diag_tilt = eval_results.diag_tilt
diag_fwd_vel = eval_results.diag_fwd_vel
diag_pelvis_h = eval_results.diag_pelvis_h
diag_l_foot = eval_results.diag_l_foot
diag_r_foot = eval_results.diag_r_foot
diag_energy = eval_results.diag_energy
diag_reward_components = eval_results.diag_reward_components

print(f"\nEvaluation results ({eval_config.n_episodes} episodes):")
print(f"  Mean reward:      {mean_reward:.2f} +/- {std_reward:.2f}")
print(f"  Mean length:      {mean_length:.1f} +/- {std_length:.1f}")
print(f"  Mean fwd vel:     {mean_fwd_vel:.3f} m/s")
print(f"  Mean distance:    {mean_distance:.2f} m")
print(f"  Mean tilt:        {np.degrees(mean_tilt):.1f} deg")
print(f"  Mean pelvis H:    {mean_height:.3f} m")

# Build stage results dict (compatible with reporting.save_results_csv)
stage_results = {
    "stage": CURRENT_STAGE,
    "name": _cfg["stage_names"].get(CURRENT_STAGE, f"Stage {CURRENT_STAGE}"),
    "description": f"JAX/MJX PPO stage {CURRENT_STAGE}",
    "timesteps": NUM_UPDATES * ROLLOUT_LEN * NUM_ENVS,
    "duration_seconds": elapsed,
    "mean_reward": round(mean_reward, 2),
    "std_reward": round(std_reward, 2),
    "mean_episode_length": round(mean_length, 1),
    "std_episode_length": round(std_length, 1),
    "mean_forward_vel": round(mean_fwd_vel, 3),
    "std_forward_vel": round(std_fwd_vel, 3),
    "mean_distance_traveled": round(mean_distance, 2),
    "best_eval_reward": round(best_reward, 2),
    "best_eval_timestep": best_update * ROLLOUT_LEN * NUM_ENVS,
    "sim_dt": mj_model.opt.timestep * FRAME_SKIP,
    "model_path": str(MODEL_DIR / "best_model"),
}

# Check gate conditions using shared function
gate_passed, gate_failures = check_stage_gate(eval_results, gate_min_reward, gate_min_length)
stage_results["gate_passed"] = gate_passed

if gate_failures:
    print(f"\n*** STAGE {CURRENT_STAGE} GATE NOT PASSED ***")
    for f in gate_failures:
        print(f"  - {f}")
    print("Do NOT proceed to the next stage. Re-train with more updates or adjusted hyperparameters.")
else:
    print(f"\n*** STAGE {CURRENT_STAGE} GATE PASSED ***")
    print("Safe to advance to the next stage.")

## Save Results & Stage Summary

Save structured training artifacts: stage summary text file, collected results
CSV (compatible with sweep analysis tooling), and model checkpoints.

In [ ]:
import csv
import json

from environments.shared.jax_checkpoint import save_checkpoint
from environments.shared.reporting import format_duration, write_stage_summary

# ============================================================
# 1. Stage summary text file (saved to stage dir)
# ============================================================
sim_dt = stage_results.get("sim_dt", mj_model.opt.timestep * FRAME_SKIP)

write_stage_summary(STAGE_DIR, stage_results, SPECIES, "JAX/MJX PPO")
print(f"Stage summary saved: {STAGE_DIR / 'stage_summary.txt'}")

# Also print the summary inline
stage_summary_path = STAGE_DIR / "stage_summary.txt"
print()
print(stage_summary_path.read_text())

# ============================================================
# 2. Save stage config snapshot
# ============================================================
config_snapshot = {
    "species": SPECIES,
    "stage": CURRENT_STAGE,
    "algorithm": "jax_ppo",
    "jax_kwargs": {
        "num_envs": NUM_ENVS,
        "rollout_len": ROLLOUT_LEN,
        "num_updates": NUM_UPDATES,
        "ppo_epochs": PPO_EPOCHS,
        "minibatch_size": MINIBATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "max_grad_norm": MAX_GRAD_NORM,
        "gamma": GAMMA,
        "gae_lambda": GAE_LAMBDA,
        "clip_range": CLIP_RANGE,
        "ent_coef": ENT_COEF,
        "fall_penalty": FALL_PENALTY,
        "reset_noise_scale": RESET_NOISE_SCALE,
        "init_qpos_noise": INIT_QPOS_NOISE,
        "init_yaw_noise": INIT_YAW_NOISE,
        "warmup_updates": WARMUP_UPDATES,
        "warmup_clip_range": WARMUP_CLIP_RANGE,
        "warmup_ent_coef": WARMUP_ENT_COEF,
        "ramp_updates": RAMP_UPDATES,
        "ramp_attr": RAMP_ATTR,
        "ramp_start_fraction": RAMP_START_FRACTION,
    },
    "reward_cfg": reward_cfg,
    "env_kwargs": dict(_env_kw),
    "curriculum_kwargs": dict(_stage_cfg.get("curriculum_kwargs", {})),
}
config_path = STAGE_DIR / "stage_config.json"
with open(config_path, "w") as f:
    json.dump(config_snapshot, f, indent=2)
print(f"Stage config saved: {config_path}")

# ============================================================
# 3. Collected results CSV (compatible with sweep tooling)
# ============================================================
csv_results_path = RUN_DIR / "collected_results.csv"
_csv_existed = csv_results_path.exists()

# Read existing rows if appending to multi-stage run
_existing_rows = []
if _csv_existed:
    with open(csv_results_path, "r") as f:
        reader = csv.DictReader(f)
        _existing_rows = [row for row in reader if int(row.get("stage", 0)) != CURRENT_STAGE]

_result_row = {
    "species": SPECIES,
    "algorithm": "jax_ppo",
    "seed": 42,
    "stage": CURRENT_STAGE,
    "best_mean_reward": stage_results["best_eval_reward"],
    "last_mean_reward": stage_results["mean_reward"],
    "last_mean_episode_length": stage_results["mean_episode_length"],
    "mean_forward_vel": stage_results["mean_forward_vel"],
    "std_forward_vel": stage_results["std_forward_vel"],
    "mean_distance_traveled": stage_results["mean_distance_traveled"],
    "training_duration_seconds": round(stage_results["duration_seconds"], 1),
    "reward_threshold": curriculum.get("min_avg_reward", ""),
    "ep_length_threshold": curriculum.get("min_avg_episode_length", ""),
    "stage_passed": stage_results["gate_passed"],
}

_all_rows = _existing_rows + [_result_row]
_all_fieldnames = list(dict.fromkeys(
    k for row in _all_rows for k in row.keys()
))
with open(csv_results_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=_all_fieldnames)
    writer.writeheader()
    writer.writerows(_all_rows)
print(f"Collected results CSV saved: {csv_results_path}")

# ============================================================
# 4. Save diagnostics.npz (for downstream visualization tooling)
# ============================================================
_diag_data = {
    "tilt_angle": np.array(diag_tilt),
    "forward_vel": np.array(diag_fwd_vel),
    "pelvis_height": np.array(diag_pelvis_h),
    "energy": np.array(diag_energy),
}
if diag_l_foot:
    _diag_data["l_foot_contact"] = np.array(diag_l_foot)
    _diag_data["r_foot_contact"] = np.array(diag_r_foot)
for comp_name, comp_vals in diag_reward_components.items():
    _diag_data[f"reward_{comp_name}"] = np.array(comp_vals)

np.savez(STAGE_DIR / "diagnostics.npz", **_diag_data)
print(f"Diagnostics saved: {STAGE_DIR / 'diagnostics.npz'}")

# ============================================================
# 5. Save best model params (using library checkpoint utility)
# ============================================================
best_model_path = MODEL_DIR / "best_model.pkl"
save_checkpoint(
    best_model_path,
    best_params if best_params is not None else jax.device_get(params),
    obs_rms=obs_rms,
    extra={"best_reward": best_reward, "best_update": best_update},
)
final_model_path = MODEL_DIR / f"stage{CURRENT_STAGE}_final.pkl"
save_checkpoint(final_model_path, jax.device_get(params), obs_rms=obs_rms)
print(f"Best model saved:  {best_model_path}")
print(f"Final model saved: {final_model_path}")

# ============================================================
# 6. Full training summary text file (run-level)
# ============================================================
training_summary_path = RUN_DIR / "training_summary.txt"
_ts_lines = [
    "Mesozoic Labs JAX/MJX Training Summary",
    "=" * 50,
    "",
    f"Species:        {SPECIES.title()}",
    f"Algorithm:      JAX/MJX PPO",
    f"Date:           {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"Seed:           42",
    f"Parallel envs:  {NUM_ENVS}",
    f"Run directory:  {RUN_DIR}",
    "",
    f"Stage {stage_results['stage']}: {stage_results['name']}",
    f"  Description:    {stage_results['description']}",
    f"  Timesteps:      {stage_results['timesteps']:,}",
    f"  Duration:       {format_duration(stage_results['duration_seconds'])}",
    f"  Final eval:     {stage_results['mean_reward']:.2f} +/- {stage_results['std_reward']:.2f}",
    f"  Avg ep length:  {stage_results['mean_episode_length']:.1f} +/- {stage_results['std_episode_length']:.1f} steps "
    f"({stage_results['mean_episode_length'] * sim_dt:.2f}s sim time)",
    f"  Avg fwd vel:    {stage_results['mean_forward_vel']:.3f} +/- {stage_results['std_forward_vel']:.3f} m/s",
    f"  Best eval:      {stage_results['best_eval_reward']:.2f} (at {stage_results['best_eval_timestep']:,} steps)",
    f"  Best model:     {best_model_path}",
    f"  Gate passed:    {stage_results['gate_passed']}",
    "",
    "-" * 50,
    f"Total training time: {format_duration(stage_results['duration_seconds'])}",
]
training_summary_text = "\n".join(_ts_lines) + "\n"
training_summary_path.write_text(training_summary_text)
print(f"\nTraining summary saved: {training_summary_path}")

## 7. Training Curves & Locomotion Diagnostics

In [ ]:
# Training curves — uses shared plot_training_curves from jax_viz
curve_path = OUTPUT_DIR / f"{SPECIES}_jax_training_curves.png"

plot_training_curves(
    reward_history=reward_history,
    loss_history=loss_history,
    episode_return_history=episode_return_history,
    diagnostics_history=diagnostics_history,
    species=SPECIES,
    stage=CURRENT_STAGE,
    output_path=curve_path,
    show=True,
)

print(f"Training curves saved to: {curve_path}")

In [ ]:
# Locomotion diagnostics — uses shared plot_locomotion_diagnostics from jax_viz

plot_locomotion_diagnostics(
    eval_results,
    species=SPECIES,
    stage=CURRENT_STAGE,
    max_tilt_angle=MAX_TILT_ANGLE,
    healthy_z_range=(HEALTHY_Z_MIN, HEALTHY_Z_MAX),
    output_dir=STAGE_DIR,
    show=True,
)

print(f"Locomotion diagnostics saved to: {STAGE_DIR}")

## 8. Record Training Video

Record a video of the best trained policy (highest reward during training) using
the CPU MuJoCo renderer. The JAX policy is evaluated deterministically (using the
action mean).

In [ ]:
try:
    import mediapy
    _HAS_MEDIAPY = True
except ImportError:
    _HAS_MEDIAPY = False
    print("mediapy not installed. Install with: pip install mediapy")

if _HAS_MEDIAPY:
    video_params = best_params if best_params is not None else jax.device_get(params)
    print(f"Recording video with best model (update {best_update}, reward {best_reward:+.4f})")

    video_path = str(OUTPUT_DIR / f"{SPECIES}_jax_mjx_training.mp4")

    frames, episode_reward = record_training_video(
        mj_model, video_params, network, obs_rms,
        get_obs_fn=get_obs,
        normalize_obs_fn=normalize_obs,
        scale_action_fn=scale_action,
        reward_fn=compute_reward,
        reward_cfg=reward_cfg,
        max_episode_steps=MAX_EPISODE_STEPS,
        frame_skip=FRAME_SKIP,
        root_body_id=ROOT_BODY_ID,
        healthy_z_range=(HEALTHY_Z_MIN, HEALTHY_Z_MAX),
        output_path=video_path,
        fps=50,
        show=True,
    )

    print(f"Episode reward: {episode_reward:.2f} | {len(frames)} frames")
    print(f"Saved to: {video_path}")

## 9. Next Steps

To continue with curriculum training, change `CURRENT_STAGE` in the configuration
cell above and re-run from there. The reward config is loaded automatically from the
TOML files in `configs/<species>/`:

```python
CURRENT_STAGE = 2  # or 3
```

The policy parameters carry over automatically between stages.
After each stage, re-run the Stage Gate Evaluation cell to verify the curriculum
thresholds are met, then re-run the video recording cell to capture the new behavior.

**Resuming from a checkpoint:** If your Colab session disconnects, set
`RESUME_FROM` in the configuration cell to reload parameters and obs stats:

```python
RESUME_FROM = "trex_jax_checkpoint_100.pkl"
```

To train a different species, change `SPECIES` in the configuration cell and
restart from the beginning.

## 10. Auto-Disconnect (Optional)

Optionally disconnect the Colab runtime after completion to free up resources.
Set `AUTO_DISCONNECT = True` in the configuration cell or toggle below.

In [ ]:
AUTO_DISCONNECT = True  # Set to True to disconnect runtime after training

if AUTO_DISCONNECT:
    import time
    print("Training finished. Disconnecting runtime in 5 seconds...")
    time.sleep(5)
    from google.colab import runtime
    runtime.unassign()
else:
    print("Training finished. Runtime kept alive — remember to disconnect manually when done.")